# 📦 ***Library & Setup***

## 🔧 ***Library***


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q torch torchvision timm scikit-learn opencv-python-headless pillow tqdm openpyxl

In [ ]:
# ============================================================
# CELL 3 — Import Libraries
# ============================================================
import numpy as np
import pandas as pd
import cv2
import time
import pickle
import warnings
warnings.filterwarnings('ignore')

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (roc_auc_score, accuracy_score,
                             confusion_matrix, ConfusionMatrixDisplay)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter

print('Libraries loaded OK')

## 🔌 ***Mount & Load Dataset***

In [ ]:
# ── Path ─────────────────────────────────────────────────
IMAGE_DIR  = "/content/drive/MyDrive/IDSC 2026/Preprocessing/3. CLAHE/CLAHE 224x224"
CSV_PATH   = "/content/drive/MyDrive/IDSC 2026/Dataset/Labels_Filtered.csv"
SAVE_DIR   = "/content/drive/MyDrive/IDSC 2026/Features"
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Config ───────────────────────────────────────────────
IMG_SIZE      = 224
BATCH_SIZE    = 64
N_FOLDS       = 5
RANDOM_STATE  = 42
AUG_PER_IMAGE = 2

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Load CSV ─────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()

print("=" * 55)
print("      🔌 DATASET LOADED")
print("=" * 55)
print(f"   Total gambar  : {len(df)}")
print(f"   GON+          : {(df['label_enc']==1).sum()}")
print(f"   GON-          : {(df['label_enc']==0).sum()}")
print(f"   Device        : {DEVICE}")
print(f"   Image DIR     : {IMAGE_DIR}")
print(f"   Save DIR      : {SAVE_DIR}")
print("=" * 55)

# ── SVM Config ───────────────────────────────────────────
SVM_KERNEL = 'rbf'
SVM_C      = 1.0
SVM_GAMMA  = 'scale'


# ✂️ ***Splitting Dataset***

## 🔀 ***Stratified Group K-Fold***

In [ ]:
X      = df["Image Name"].values
y      = df["label_enc"].values
groups = df["Patient"].values

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

df["fold"] = -1
for fold, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups)):
    df.loc[val_idx, "fold"] = fold

print("=" * 65)
print(f"     🔀 STRATIFIED GROUP K-FOLD  —  K = {N_FOLDS}")
print("=" * 65)
print(f"\n{'Fold':<6} {'Train':>6} {'Val':>6} {'GON+ Tr':>10} {'GON- Tr':>10} {'GON+ Val':>10} {'GON- Val':>10}")
print("-" * 65)

for fold in range(N_FOLDS):
    tr = df[df["fold"] != fold]
    vl = df[df["fold"] == fold]
    print(f"{fold:<6} {len(tr):>6} {len(vl):>6} "
          f"{(tr['label_enc']==1).sum():>10} "
          f"{(tr['label_enc']==0).sum():>10} "
          f"{(vl['label_enc']==1).sum():>10} "
          f"{(vl['label_enc']==0).sum():>10}")

print("-" * 65)

## 🔍 ***Cek Data Leakage***
python# ============================================================
# CELL 5 — Cek Data Leakage
# ============================================================
print("=" * 55)
print("     🔍 CEK DATA LEAKAGE ANTAR FOLD")
print("=" * 55)

all_aman = True
for fold in range(N_FOLDS):
    train_patients = set(df[df["fold"] != fold]["Patient"])
    val_patients   = set(df[df["fold"] == fold]["Patient"])
    bocor          = train_patients & val_patients
    status         = "✅ Aman" if len(bocor) == 0 else f"❌ Bocor {len(bocor)} pasien!"
    if len(bocor) > 0:
        all_aman = False
    print(f"   Fold {fold} : {status}")

print("-" * 55)
print(f"\n{'🎉 Semua fold aman!' if all_aman else '⚠️ Ada leakage!'}")

# 🗃️ ***Dataset & Augmentasi***

## 🖼️ ***Custom Dataset Class***

In [ ]:
def preprocess_fundus(img_path, img_size=224, sigma=30):
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        return Image.fromarray(np.zeros((img_size, img_size, 3), dtype=np.uint8))

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w    = img_rgb.shape[:2]

    # ── Circle Crop ──────────────────────────────────────
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    cropped = None
    for thresh_val in [10, 5, 15, 20]:
        _, mask = cv2.threshold(gray, thresh_val, 255, cv2.THRESH_BINARY)
        coords  = cv2.findNonZero(mask)
        if coords is not None:
            x, y, cw, ch = cv2.boundingRect(coords)
            if cw > w * 0.3 and ch > h * 0.3:
                pad = 5
                x1  = max(0, x - pad)
                y1  = max(0, y - pad)
                x2  = min(w, x + cw + pad)
                y2  = min(h, y + ch + pad)
                cropped = img_rgb[y1:y2, x1:x2]
                break
    if cropped is None:
        cropped = img_rgb

    # ── Ben Graham ────────────────────────────────────────
    img_float = cropped.astype(np.float32)
    blur      = cv2.GaussianBlur(img_float, (0, 0), sigmaX=sigma)
    img_bg    = cv2.addWeighted(img_float, 4, blur, -4, 128)
    img_bg    = np.clip(img_bg, 0, 255).astype(np.uint8)

    # ── CLAHE Green Channel ───────────────────────────────
    clahe   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    r, g, b = cv2.split(img_bg)
    g_clahe = clahe.apply(g)
    img_enh = cv2.merge([r, g_clahe, b])

    # ── Resize ────────────────────────────────────────────
    img_resized = cv2.resize(img_enh, (img_size, img_size),
                             interpolation=cv2.INTER_AREA)
    return Image.fromarray(img_resized)

print('✅ Preprocessing function siap')

## 🔁 ***Augmentasi***

In [ ]:
AUG_TRANSFORMS = [
    T.RandomRotation(degrees=15),
    T.ColorJitter(brightness=0.3, contrast=0.3),
]
AUG_TRANSFORMS = AUG_TRANSFORMS[:AUG_PER_IMAGE]

to_tensor_norm = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

class FundusDataset(Dataset):
    def __init__(self, df, image_dir, img_size=224, is_train=False):
        self.df        = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.img_size  = img_size
        self.samples   = []
        for i in range(len(self.df)):
            self.samples.append((i, 0))
            if is_train:
                for a in range(len(AUG_TRANSFORMS)):
                    self.samples.append((i, a + 1))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        row_idx, aug_idx = self.samples[idx]
        row      = self.df.iloc[row_idx]
        img_path = os.path.join(self.image_dir, row['Image Name'])
        img      = preprocess_fundus(img_path, self.img_size)
        if aug_idx > 0:
            img = AUG_TRANSFORMS[aug_idx - 1](img)
        img   = to_tensor_norm(img)
        label = int(row['label_enc'])
        return img, label

print(f"✅ Dataset class siap")
print(f"   Aug per gambar : {AUG_PER_IMAGE}")
print(f"   Total aug      : 1 original + {AUG_PER_IMAGE} augmentasi = {1+AUG_PER_IMAGE} per gambar train")

# 🏗️ ***Feature Extractor***


## 🏗️ ***Build CNN Frozen***


In [ ]:
def build_feature_extractor(model_name, device):
    backbone = timm.create_model(model_name, pretrained=True, num_classes=0)

    for param in backbone.parameters():
        param.requires_grad = False

    backbone.eval().to(device)

    with torch.no_grad():
        dummy    = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE).to(device)
        feat_dim = backbone(dummy).shape[1]

    total_params = sum(p.numel() for p in backbone.parameters())
    print(f"   Backbone  : {model_name}")
    print(f"   Feat dim  : {feat_dim:,}")
    print(f"   Params    : {total_params:,} (semua frozen)")
    return backbone, feat_dim

print("✅ build_feature_extractor siap")

## 📤 ***Fungsi Extract Features***


In [ ]:
@torch.no_grad()
def extract_and_save(model_name, df_data, save_subdir):
    save_path = os.path.join(SAVE_DIR, save_subdir)
    os.makedirs(save_path, exist_ok=True)

    patients   = df_data['Patient'].values
    labels_arr = df_data['label_enc'].values
    sgkf_      = StratifiedGroupKFold(n_splits=N_FOLDS,
                                      shuffle=True, random_state=RANDOM_STATE)

    print(f"\n{'='*62}")
    print(f"  📤 Ekstraksi Fitur — {model_name}")
    print(f"  📁 Simpan ke       — {save_path}")
    print(f"{'='*62}")

    backbone, feat_dim = build_feature_extractor(model_name, DEVICE)

    for fold_idx, (train_idx, val_idx) in enumerate(
            sgkf_.split(df_data, labels_arr, groups=patients)):

        print(f"\n  ── Fold {fold_idx+1}/{N_FOLDS} ─────────────────────")

        df_tr = df_data.iloc[train_idx].reset_index(drop=True)
        df_vl = df_data.iloc[val_idx].reset_index(drop=True)

        train_ds = FundusDataset(df_tr, IMAGE_DIR, IMG_SIZE, is_train=True)
        val_ds   = FundusDataset(df_vl, IMAGE_DIR, IMG_SIZE, is_train=False)

        train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=2)
        val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=2)

        # Ekstrak train
        all_feats, all_labels = [], []
        t0 = time.time()
        for imgs, labels in tqdm(train_dl, desc=f"  Train fold {fold_idx+1}"):
            feats = backbone(imgs.to(DEVICE)).cpu().numpy()
            all_feats.append(feats)
            all_labels.extend(labels.numpy())
        X_train = np.vstack(all_feats)
        y_train = np.array(all_labels)
        t_train = time.time() - t0
        print(f"  Train → {X_train.shape} | {t_train:.2f}s")

        # Ekstrak val
        all_feats, all_labels = [], []
        t0 = time.time()
        for imgs, labels in tqdm(val_dl, desc=f"  Val   fold {fold_idx+1}"):
            feats = backbone(imgs.to(DEVICE)).cpu().numpy()
            all_feats.append(feats)
            all_labels.extend(labels.numpy())
        X_val = np.vstack(all_feats)
        y_val = np.array(all_labels)
        t_val = time.time() - t0
        print(f"  Val   → {X_val.shape} | {t_val:.2f}s")

        # Simpan per fold
        fold_dir = os.path.join(save_path, f"fold_{fold_idx}")
        os.makedirs(fold_dir, exist_ok=True)

        np.save(os.path.join(fold_dir, "X_train.npy"), X_train)
        np.save(os.path.join(fold_dir, "y_train.npy"), y_train)
        np.save(os.path.join(fold_dir, "X_val.npy"),   X_val)
        np.save(os.path.join(fold_dir, "y_val.npy"),   y_val)

        print(f"  💾 Tersimpan: {fold_dir}")

    print(f"\n✅ {model_name} — semua fold selesai!")

print("✅ extract_and_save siap")

# 🔵 ***EfficientNet-B0***


In [ ]:
print("=" * 55)
print("      🔵 EFFICIENTNET-B0")
print("=" * 55)

extract_and_save(
    model_name  = "efficientnet_b0",
    df_data     = df,
    save_subdir = "EfNetB0"
)

# 🟢 ***EfficientNet-B3***


In [ ]:
print("=" * 55)
print("      🟢 EFFICIENTNET-B3")
print("=" * 55)

extract_and_save(
    model_name  = "efficientnet_b3",
    df_data     = df,
    save_subdir = "EfNetB3"
)

# 🔴 ***ResNet-50***


In [ ]:
print("=" * 55)
print("      🔴 RESNET-50")
print("=" * 55)

extract_and_save(
    model_name  = "resnet50",
    df_data     = df,
    save_subdir = "ResNet50"
)

# ✅ ***Verifikasi Hasil Ekstraksi***


In [ ]:
print("=" * 65)
print("      ✅ VERIFIKASI HASIL EKSTRAKSI")
print("=" * 65)

models_check = {
    "🔵 EfNetB0" : "EfNetB0",
    "🟢 EfNetB3" : "EfNetB3",
    "🔴 ResNet50": "ResNet50",
}

for model_label, subdir in models_check.items():
    print(f"\n  {model_label}")
    for fold in range(N_FOLDS):
        fold_dir = os.path.join(SAVE_DIR, subdir, f"fold_{fold}")
        if os.path.exists(fold_dir):
            X_tr = np.load(os.path.join(fold_dir, "X_train.npy"))
            X_vl = np.load(os.path.join(fold_dir, "X_val.npy"))
            nan_tr = np.isnan(X_tr).sum()
            nan_vl = np.isnan(X_vl).sum()
            status = "✅" if nan_tr == 0 and nan_vl == 0 else "⚠️ Ada NaN!"
            print(f"    Fold {fold} {status} | Train {X_tr.shape} | Val {X_vl.shape}")
        else:
            print(f"    Fold {fold} ❌ folder tidak ditemukan!")

print("\n" + "=" * 65)
print("   ✅ Semua fitur siap untuk Classification!")
print("=" * 65)

# 🤖 ***SVM Classifier***


## ⚙️ ***Feature Extractor, Evaluasi & Fungsi SVM***


In [ ]:
# ============================================================
# CELL 7 — CNN Feature Extractor (Frozen) & Fungsi Evaluasi
# ============================================================

# ── A. Build backbone frozen ──────────────────────────────────────────────────
def build_feature_extractor(model_name, device):
    """
    Load CNN pretrained ImageNet, freeze semua bobot.
    Tidak ada fine-tuning — digunakan murni untuk ekstraksi fitur.
    """
    backbone = timm.create_model(model_name, pretrained=True, num_classes=0)

    # Freeze semua parameter
    for param in backbone.parameters():
        param.requires_grad = False

    backbone.eval().to(device)

    with torch.no_grad():
        dummy    = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE).to(device)
        feat_dim = backbone(dummy).shape[1]

    total_params = sum(p.numel() for p in backbone.parameters())
    print(f'  Backbone : {model_name}')
    print(f'  Feat dim : {feat_dim:,}')
    print(f'  Params   : {total_params:,} (semua frozen)')
    return backbone, feat_dim


# ── B. Ekstraksi fitur — catat waktu secara terpisah ─────────────────────────
@torch.no_grad()
def extract_features(backbone, dataloader, device, split_name=''):
    """
    Ekstrak feature vector dari seluruh gambar di dataloader.
    Return: features (N x D), labels (N,), waktu_detik (float)
    """
    backbone.eval()
    all_feats, all_labels = [], []

    t0 = time.time()
    for imgs, labels in tqdm(dataloader,
                             desc=f'  Ekstrak {split_name}', leave=False):
        feats = backbone(imgs.to(device)).cpu().numpy()
        all_feats.append(feats)
        all_labels.extend(labels.numpy())
    elapsed = time.time() - t0

    return np.vstack(all_feats), np.array(all_labels), elapsed


# ── C. Hitung metrik evaluasi ─────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_prob):
    """Return: accuracy, sensitivity, specificity, auc, confusion_matrix"""
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)

    acc  = accuracy_score(y_true, y_pred)
    sen  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spe  = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    try:    auc = roc_auc_score(y_true, y_prob)
    except: auc = 0.0

    return {'accuracy': acc, 'sensitivity': sen,
            'specificity': spe, 'auc': auc, 'cm': cm}


# ── D. Plot confusion matrix per fold ────────────────────────────────────────
def plot_confusion_matrices(fold_results, model_name, save_path=None):
    n     = len(fold_results)
    ncols = min(n, 3)
    nrows = (n + ncols - 1) // ncols + 1   # +1 baris rata-rata

    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(5 * ncols, 5 * nrows))
    axes = axes.flatten()

    for i, res in enumerate(fold_results):
        disp = ConfusionMatrixDisplay(res['cm'],
                                     display_labels=['GON-', 'GON+'])
        disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
        axes[i].set_title(
            f'Fold {i+1}\n'
            f'Acc={res["accuracy"]:.3f}  Sen={res["sensitivity"]:.3f}\n'
            f'Spe={res["specificity"]:.3f}  AUC={res["auc"]:.3f}',
            fontsize=9)

    # Rata-rata CM
    avg_cm = np.mean([r['cm'] for r in fold_results], axis=0).astype(int)
    ConfusionMatrixDisplay(avg_cm,
                           display_labels=['GON-', 'GON+']).plot(
        ax=axes[n], colorbar=False, cmap='Oranges')
    axes[n].set_title('Rata-rata\nConfusion Matrix',
                      fontsize=10, fontweight='bold')

    for j in range(n + 1, len(axes)):
        axes[j].axis('off')

    fig.suptitle(f'{model_name} + SVM — Confusion Matrix per Fold',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()

    if save_path is None:
        save_path = f'/content/{model_name}_confusion_matrix.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  Confusion matrix disimpan: {save_path}')
    return save_path


print('Feature extractor, metrik, & confusion matrix functions siap.')
print('Mode: CNN FROZEN — tidak ada backprop / fine-tuning')

## 🔁 ***5-Fold CV — CNN Feature Extraction + SVM***


In [ ]:
# ============================================================
# CELL 8 — Fungsi 5-Fold CV (CNN Feature Extraction + SVM)
# ============================================================
def run_kfold_svm(model_name, df_data, n_folds=N_FOLDS):
    """
    Pipeline per fold:
      1. Ekstrak fitur train  → catat t_feat_train
      2. Ekstrak fitur val    → catat t_feat_val
      3. Train SVM            → catat t_svm_train
      4. Prediksi & evaluasi
    Setiap fold menyimpan:
      waktu_feat_train_s, waktu_feat_val_s, waktu_svm_s, waktu_total_s
    """
    patients   = df_data['Patient'].values
    labels_arr = df_data['label_enc'].values
    sgkf = StratifiedGroupKFold(n_splits=n_folds,
                                shuffle=True, random_state=42)

    print(f'\n{"="*62}')
    print(f'  Model  : {model_name}  (frozen CNN + SVM)')
    print(f'  SVM    : kernel={SVM_KERNEL}  C={SVM_C}  gamma={SVM_GAMMA}')
    print(f'  Folds  : {n_folds}')
    print(f'{"="*62}')

    # Build feature extractor sekali, dipakai semua fold
    backbone, feat_dim = build_feature_extractor(model_name, DEVICE)

    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(
            sgkf.split(df_data, labels_arr, groups=patients)):

        print(f'\n  ── Fold {fold_idx+1}/{n_folds} '
              f'─────────────────────────────────')

        df_tr = df_data.iloc[train_idx].reset_index(drop=True)
        df_vl = df_data.iloc[val_idx].reset_index(drop=True)

        train_ds = FundusDataset(df_tr, IMAGE_DIR, IMG_SIZE, is_train=True)
        val_ds   = FundusDataset(df_vl, IMAGE_DIR, IMG_SIZE, is_train=False)
        print(f'  Sampel train (dgn aug): {len(train_ds):,} | val: {len(val_ds):,}')

        train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=2, pin_memory=False)
        val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=2, pin_memory=False)

        # ── 1. Ekstraksi fitur train ──────────────────────────────────────
        print('  [1/3] Ekstraksi fitur train...')
        X_train, y_train, t_feat_train = extract_features(
            backbone, train_dl, DEVICE, split_name='train')
        print(f'        Shape: {X_train.shape}  |  Waktu: {t_feat_train:.2f}s')

        # ── 2. Ekstraksi fitur val ────────────────────────────────────────
        print('  [2/3] Ekstraksi fitur val...')
        X_val, y_val, t_feat_val = extract_features(
            backbone, val_dl, DEVICE, split_name='val')
        print(f'        Shape: {X_val.shape}   |  Waktu: {t_feat_val:.2f}s')

        # ── 3. Training SVM ───────────────────────────────────────────────
        print(f'  [3/3] Training SVM (C={SVM_C}, kernel={SVM_KERNEL})...')
        svm_pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('svm', SVC(C=SVM_C, kernel=SVM_KERNEL, gamma=SVM_GAMMA,
                        probability=True, random_state=42))
        ])
        t_svm_start = time.time()
        svm_pipeline.fit(X_train, y_train)
        t_svm = time.time() - t_svm_start
        print(f'        SVM selesai  |  Waktu: {t_svm:.2f}s')

        # ── 4. Prediksi & metrik ──────────────────────────────────────────
        y_pred = svm_pipeline.predict(X_val)
        y_prob = svm_pipeline.predict_proba(X_val)[:, 1]  # prob GON+

        metrics = compute_metrics(y_val, y_pred, y_prob)

        t_feat_total = t_feat_train + t_feat_val
        t_total      = t_feat_total + t_svm

        metrics['waktu_feat_train_s'] = t_feat_train
        metrics['waktu_feat_val_s']   = t_feat_val
        metrics['waktu_feat_total_s'] = t_feat_total
        metrics['waktu_svm_s']        = t_svm
        metrics['waktu_total_s']      = t_total

        print(f'\n  Fold {fold_idx+1} Hasil:')
        print(f'    Acc={metrics["accuracy"]:.4f}  '
              f'Sen={metrics["sensitivity"]:.4f}  '
              f'Spe={metrics["specificity"]:.4f}  '
              f'AUC={metrics["auc"]:.4f}')
        print(f'    Waktu → Feat-train: {t_feat_train:.2f}s | '
              f'Feat-val: {t_feat_val:.2f}s | '
              f'SVM: {t_svm:.2f}s | '
              f'Total: {t_total:.2f}s')

        fold_results.append(metrics)

        # Simpan SVM pipeline
        pkl_path = os.path.join(
            SAVE_DIR, f'{model_name}_svm_fold{fold_idx+1}.pkl')
        with open(pkl_path, 'wb') as f:
            pickle.dump(svm_pipeline, f)

    # ── Summary rata-rata ────────────────────────────────────────────────
    print(f'\n  {"─"*55}')
    print(f'  RATA-RATA  {model_name} + SVM:')
    for key in ['accuracy','sensitivity','specificity','auc',
                'waktu_feat_train_s','waktu_feat_val_s',
                'waktu_feat_total_s','waktu_svm_s','waktu_total_s']:
        vals = [r[key] for r in fold_results]
        print(f'    {key:<25}: {np.mean(vals):.4f} ± {np.std(vals):.4f}')

    return fold_results


print('Fungsi run_kfold_svm siap.')

# 🔵 ***EfficientNet-B0 + SVM***


In [ ]:
MODEL_NAME = 'efficientnet_b0'

fold_results_b0 = run_kfold_svm(MODEL_NAME, df)
cm_path_b0 = plot_confusion_matrices(fold_results_b0, MODEL_NAME)
print(f'\n✅ EfficientNet-B0 + SVM selesai!')

# 🟢 ***EfficientNet-B3 + SVM***


In [ ]:
MODEL_NAME = 'efficientnet_b3'

fold_results_b3 = run_kfold_svm(MODEL_NAME, df)
cm_path_b3 = plot_confusion_matrices(fold_results_b3, MODEL_NAME)
print(f'\n✅ EfficientNet-B3 + SVM selesai!')

# 🔴 ***ResNet-50 + SVM***


In [ ]:
MODEL_NAME = 'resnet50'

fold_results_r50 = run_kfold_svm(MODEL_NAME, df)
cm_path_r50 = plot_confusion_matrices(fold_results_r50, MODEL_NAME)
print(f'\n✅ ResNet-50 + SVM selesai!')

# 📊 ***Evaluasi Model***


## 💾 ***Simpan Evaluasi ke Excel***


In [ ]:
# ============================================================
# CELL 12 — Simpan Evaluasi ke Excel
# ============================================================
EXCEL_PATH = '/content/evaluasi_cnn_svm_sigmoid.xlsx'

# ── Helper styling ────────────────────────────────────────────────────────────
def style_header(cell, bg='1F497D'):
    cell.font      = Font(bold=True, color='FFFFFF', size=11)
    cell.fill      = PatternFill('solid', fgColor=bg)
    cell.alignment = Alignment(horizontal='center', vertical='center',
                               wrap_text=True)

def style_val(cell, bold=False):
    cell.alignment = Alignment(horizontal='center', vertical='center')
    if bold: cell.font = Font(bold=True)

def tborder():
    s = Side(border_style='thin', color='999999')
    return Border(left=s, right=s, top=s, bottom=s)

wb = openpyxl.Workbook()
wb.remove(wb.active)

# Kolom evaluasi utama
EVAL_COLS = ['Fold', 'Accuracy', 'Sensitivity', 'Specificity', 'AUC',
             'Waktu Feat-Train (s)', 'Waktu Feat-Val (s)',
             'Waktu Feat-Total (s)', 'Waktu SVM (s)', 'Waktu Total (s)']
EVAL_KEYS = ['accuracy', 'sensitivity', 'specificity', 'auc',
             'waktu_feat_train_s', 'waktu_feat_val_s',
             'waktu_feat_total_s', 'waktu_svm_s', 'waktu_total_s']

ALL_RESULTS = {
    'EfficientNet-B0': fold_results_b0,
    'EfficientNet-B3': fold_results_b3,
    'ResNet-50':       fold_results_r50,
}
HDR_COLORS = {
    'EfficientNet-B0': '2E75B6',
    'EfficientNet-B3': '375623',
    'ResNet-50':       '843C0C',
}

# ── Sheet per model ───────────────────────────────────────────────────────────
for model_name, fold_results in ALL_RESULTS.items():
    ws = wb.create_sheet(title=model_name)
    hc = HDR_COLORS.get(model_name, '1F497D')
    n_cols = len(EVAL_COLS)

    # Judul
    ws.merge_cells(f'A1:{get_column_letter(n_cols)}1')
    ws['A1'] = f'Evaluasi {model_name} + SVM (sigmoid, C=1) — {N_FOLDS}-Fold CV'
    ws['A1'].font      = Font(bold=True, size=13, color='FFFFFF')
    ws['A1'].fill      = PatternFill('solid', fgColor=hc)
    ws['A1'].alignment = Alignment(horizontal='center', vertical='center')
    ws.row_dimensions[1].height = 28

    # Sub-header grup waktu
    ws.merge_cells('B2:E2')
    ws['B2'] = 'Metrik Klasifikasi'
    style_header(ws['B2'], bg=hc)

    ws.merge_cells('F2:J2')
    ws['F2'] = 'Waktu Komputasi (detik)'
    style_header(ws['F2'], bg='44546A')

    # Header kolom (baris 3)
    for col_i, col_name in enumerate(EVAL_COLS, start=1):
        cell = ws.cell(row=3, column=col_i, value=col_name)
        style_header(cell, bg=hc if col_i <= 5 else '44546A')
        ws.column_dimensions[get_column_letter(col_i)].width = 20
    ws['A2'] = ''

    # Data per fold
    means = {k: [] for k in EVAL_KEYS}
    for fold_idx, res in enumerate(fold_results):
        row = fold_idx + 4
        ws.cell(row=row, column=1, value=f'Fold {fold_idx+1}')
        style_val(ws.cell(row=row, column=1))
        for col_i, key in enumerate(EVAL_KEYS, start=2):
            val  = round(float(res[key]), 4)
            cell = ws.cell(row=row, column=col_i, value=val)
            style_val(cell)
            cell.border = tborder()
            means[key].append(float(res[key]))

    # Baris rata-rata
    avg_row = N_FOLDS + 4
    ws.cell(row=avg_row, column=1, value='Rata-rata')
    style_val(ws.cell(row=avg_row, column=1), bold=True)
    ws.cell(row=avg_row, column=1).fill = PatternFill('solid', fgColor='D6DCE4')
    for col_i, key in enumerate(EVAL_KEYS, start=2):
        cell = ws.cell(row=avg_row, column=col_i,
                       value=f'{np.mean(means[key]):.4f} ± {np.std(means[key]):.4f}')
        style_val(cell, bold=True)
        cell.fill   = PatternFill('solid', fgColor='D6DCE4')
        cell.border = tborder()

    for r in range(2, avg_row + 1):
        for c in range(1, n_cols + 1):
            ws.cell(row=r, column=c).border = tborder()


# ── Sheet Ringkasan semua model ───────────────────────────────────────────────
ws_s = wb.create_sheet(title='Ringkasan', index=0)
SUM_COLS = ['Model', 'Fold', 'Accuracy', 'Sensitivity', 'Specificity', 'AUC',
            'Feat-Train (s)', 'Feat-Val (s)', 'Feat-Total (s)',
            'SVM Train (s)', 'Waktu Total (s)']
SUM_KEYS = EVAL_KEYS  # sama urutan

ws_s.merge_cells(f'A1:{get_column_letter(len(SUM_COLS))}1')
ws_s['A1'] = 'Ringkasan Evaluasi — CNN (Frozen) + SVM (sigmoid, C=1) | 3 Model'
ws_s['A1'].font      = Font(bold=True, size=13, color='FFFFFF')
ws_s['A1'].fill      = PatternFill('solid', fgColor='44546A')
ws_s['A1'].alignment = Alignment(horizontal='center', vertical='center')
ws_s.row_dimensions[1].height = 28

for ci, cn in enumerate(SUM_COLS, 1):
    cell = ws_s.cell(row=2, column=ci, value=cn)
    style_header(cell, bg='44546A')
    ws_s.column_dimensions[get_column_letter(ci)].width = 18

row_cur = 3
ROW_CLR = ['EBF3FB', 'FEF2E8', 'E9F2E9']
for m_i, (mname, fres) in enumerate(ALL_RESULTS.items()):
    fc  = ROW_CLR[m_i % 3]
    mns = {k: [] for k in SUM_KEYS}
    for fi, res in enumerate(fres):
        ws_s.cell(row=row_cur, column=1, value=mname if fi == 0 else '')
        ws_s.cell(row=row_cur, column=2, value=f'Fold {fi+1}')
        for ci, key in enumerate(SUM_KEYS, 3):
            cell = ws_s.cell(row=row_cur, column=ci,
                             value=round(float(res[key]), 4))
            cell.alignment = Alignment(horizontal='center')
            cell.fill      = PatternFill('solid', fgColor=fc)
            cell.border    = tborder()
            mns[key].append(float(res[key]))
        row_cur += 1
    # Baris rata-rata
    ws_s.cell(row=row_cur, column=1, value=mname)
    ws_s.cell(row=row_cur, column=2, value='Rata-rata')
    ws_s.cell(row=row_cur, column=1).font = Font(bold=True)
    ws_s.cell(row=row_cur, column=2).font = Font(bold=True)
    for ci, key in enumerate(SUM_KEYS, 3):
        cell = ws_s.cell(row=row_cur, column=ci,
                         value=f'{np.mean(mns[key]):.4f} ± {np.std(mns[key]):.4f}')
        cell.font      = Font(bold=True)
        cell.alignment = Alignment(horizontal='center')
        cell.fill      = PatternFill('solid', fgColor='D6DCE4')
        cell.border    = tborder()
    row_cur += 2

for r in range(2, row_cur):
    for c in range(1, len(SUM_COLS)+1):
        ws_s.cell(row=r, column=c).border = tborder()

wb.save(EXCEL_PATH)
print(f'Excel disimpan: {EXCEL_PATH}')
print('Sheet: Ringkasan + EfficientNet-B0, EfficientNet-B3, ResNet-50')
print('Kolom waktu: Feat-Train | Feat-Val | Feat-Total | SVM Train | Total')

## ☁️ ***Copy Output ke Google Drive***


In [ ]:
# ============================================================
# CELL 13 — Copy Semua Output ke Google Drive
# ============================================================
import shutil, glob
DRIVE_OUT = os.path.join(BASE_DIR, 'Output_CNN_SVM/sigmoid')
os.makedirs(DRIVE_OUT, exist_ok=True)
files_to_copy = [
    '/content/evaluasi_cnn_svm_sigmoid.xlsx',
    '/content/augmentation_demo.png',
    '/content/efficientnet_b0_confusion_matrix.png',
    '/content/efficientnet_b3_confusion_matrix.png',
    '/content/resnet50_confusion_matrix.png',
]
for src in files_to_copy:
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(DRIVE_OUT, os.path.basename(src)))
        print(f'OK : {os.path.basename(src)}')
    else:
        print(f'SKIP: {src}')
# Salin SVM models (.pkl)
for pkl in glob.glob('/content/models/*.pkl'):
    shutil.copy2(pkl, os.path.join(DRIVE_OUT, os.path.basename(pkl)))
    print(f'OK : {os.path.basename(pkl)}')
print(f'\nSemua output disimpan ke: {DRIVE_OUT}')
print('Selesai!')